In [1]:
import pandas as pd
import json
import plotly.graph_objects as go
from pathlib import Path
base = Path.cwd().parent


# Label Verfication

You should document: a table or histogram showing the microbleed count distribution across 
subjects (e.g., how many subjects have 0, 1–5, 5–20, or 20+ microbleeds), and a histogram of 
individual microbleed sizes in mm³ (converting from voxel count using the voxel spacing from the 
NIfTI header). These statistics belong in the shared baseline section of their final report. 

In [2]:
path = base / "data" / "nnUNet_raw" / "Dataset001_VALDO" / "verification_stats" / "stats.json"
with open(path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# mextracting cmb counts
counts = [
    subj["microbleed_count"]
    for subj in data["statistics"].values()
]

# converting to categories
bins = {
    "0": 0,
    "1–5": 0,
    "6–20": 0,
    "20+": 0
}

for c in counts:
    if c == 0:
        bins["0"] += 1
    elif 1 <= c <= 5:
        bins["1–5"] += 1
    elif 6 <= c <= 20:
        bins["6–20"] += 1
    else:
        bins["20+"] += 1

In [3]:
fig = go.Figure([
    go.Bar(
        x=list(bins.keys()),
        y=list(bins.values())
    )
])

fig.update_layout(
    template="plotly_white",
    xaxis_title="Microbleed Count Range",
    yaxis_title="Number of Subjects"
)

fig.show()
fig.write_image("cmb_distr_across_subj.png", width=1600, height=900, scale=3)


In [11]:
volumes = []

for subj in data["statistics"].values():
    volumes.extend(subj["microbleed_vol_mm3"])

import plotly.express as px

fig = px.histogram(
    volumes,
    nbins=30,
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Microbleed Volume (mm³)",
    yaxis_title="Frequency",
    showlegend=False
)

fig.show()
fig.write_image("cmb_vol_distr.png", width=1600, height=600, scale=3)


In [12]:
df = pd.DataFrame(list(bins.items()), columns=["Range", "Subjects"])
print(df)

  Range  Subjects
0     0        22
1   1–5        40
2  6–20         7
3   20+         3


# Evaluation Results

In [28]:
path = base / "data" / "nnUNet_inference" / "evaluation_results.csv"
df = pd.read_csv(path)
df.describe()

,sensitivity,false_positives,f1_score,dice,num_gt_microbleeds,num_pred_microbleeds
count,72.000000,72.0,72.000000,72.000000,72.000000,72.000000
mean,0.960042,0.0,0.970842,0.895497,3.277778,2.944444
std,0.140773,0.0,0.126630,0.159776,9.464868,8.244313
min,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.0,1.000000,0.830152,0.000000,0.000000
50%,1.000000,0.0,1.000000,0.977396,1.000000,1.000000
75%,1.000000,0.0,1.000000,1.000000,2.000000,2.000000
max,1.000000,0.0,1.000000,1.000000,73.000000,63.000000


In [29]:
metrics_cols = ['sensitivity', 'false_positives', 'f1_score', 'dice', 'num_gt_microbleeds', 'num_pred_microbleeds']
summary_df = df[metrics_cols].describe().round(3).reset_index()
summary_df.rename(columns={'index': 'Statistic'}, inplace=True)

# Metrics Chart
df_melted = df.melt(id_vars=['subject_id'], 
                    value_vars=['sensitivity', 'f1_score', 'dice'], 
                    var_name='Metric', 
                    value_name='Score')

df_melted['Metric'] = df_melted['Metric'].replace({
    'sensitivity': 'Sensitivity',
    'f1_score': 'F1 Score',
    'dice': 'Dice Coefficient'
})

fig_box = px.box(df_melted, x='Metric', y='Score', color='Metric', 
    points='all',
    hover_data=['subject_id'])

fig_box.update_layout(yaxis_title='Score', showlegend=False, template="plotly_white")
fig_box.show()
fig_box.write_image("eval_metrics_boxplot.png", width=1600, height=600, scale=3)


# Ground Truth vs. Predicted Microbleeds
max_val = max(df['num_gt_microbleeds'].max(), df['num_pred_microbleeds'].max())

fig_scatter = px.scatter(df, x='num_gt_microbleeds', y='num_pred_microbleeds', 
                         hover_data=['subject_id'],
                         labels={
                             'num_gt_microbleeds': 'Ground Truth Microbleeds',
                             'num_pred_microbleeds': 'Predicted Microbleeds'
                         },
                         opacity=0.7)

fig_scatter.add_trace(go.Scatter(x=[0, max_val], y=[0, max_val], 
                                 mode='lines', 
                                 name='Perfect Agreement (y=x)',
                                 line=dict(color='red', dash='dash')))

fig_scatter.update_layout(xaxis=dict(range=[-1, max_val+2]), 
                          yaxis=dict(range=[-1, max_val+2]), template="plotly_white")
fig_scatter.show()
fig_scatter.write_image("gt_pred_scatter.png", width=1600, height=600, scale=3)
